In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
import json
import pickle
import random
import numpy as np
import pandas as pd

from pathlib import Path
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    recall_score,
    classification_report,
    confusion_matrix
)

SEEDS = [42, 123, 2024]

BASE_PROJECT = Path("/content/drive/MyDrive/New Jurnal Cross")

FEATURE_DIR = BASE_PROJECT / "processed_intra_features_hc_noaug"
OUT_DIR = BASE_PROJECT / "results_intra_handcrafted_mlp"
OUT_DIR.mkdir(parents=True, exist_ok=True)

DATASETS = ["emodb", "ravdess", "resd"]

LABELS = [
    "angry",
    "disgust",
    "fear",
    "happy",
    "neutral",
    "sad"
]

ID_TO_LABEL = {i: label for i, label in enumerate(LABELS)}
LABEL_TO_ID = {label: i for i, label in ID_TO_LABEL.items()}

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE:", DEVICE)
print("FEATURE_DIR:", FEATURE_DIR)
print("OUT_DIR:", OUT_DIR)

for ds in DATASETS:
    print(ds, (FEATURE_DIR / ds).exists())

DEVICE: cuda
FEATURE_DIR: /content/drive/MyDrive/New Jurnal Cross/processed_intra_features_hc_noaug
OUT_DIR: /content/drive/MyDrive/New Jurnal Cross/results_intra_handcrafted_mlp
emodb True
ravdess True
resd True


In [3]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def load_dataset_features(dataset_name):
    ds_dir = FEATURE_DIR / dataset_name

    X_train = np.load(ds_dir / "X_hc_train.npy").astype(np.float32)
    y_train = np.load(ds_dir / "y_train.npy").astype(np.int64)

    X_val = np.load(ds_dir / "X_hc_val.npy").astype(np.float32)
    y_val = np.load(ds_dir / "y_val.npy").astype(np.int64)

    X_test = np.load(ds_dir / "X_hc_test.npy").astype(np.float32)
    y_test = np.load(ds_dir / "y_test.npy").astype(np.int64)

    meta_train = pd.read_csv(ds_dir / "meta_train.csv")
    meta_val = pd.read_csv(ds_dir / "meta_val.csv")
    meta_test = pd.read_csv(ds_dir / "meta_test.csv")

    return {
        "X_train": X_train,
        "y_train": y_train,
        "X_val": X_val,
        "y_val": y_val,
        "X_test": X_test,
        "y_test": y_test,
        "meta_train": meta_train,
        "meta_val": meta_val,
        "meta_test": meta_test,
    }


def compute_metrics(y_true, y_pred):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "weighted_f1": f1_score(y_true, y_pred, average="weighted", zero_division=0),
        "uar": recall_score(y_true, y_pred, average="macro", zero_division=0),
    }


def make_report_df(y_true, y_pred):
    report = classification_report(
        y_true,
        y_pred,
        target_names=LABELS,
        labels=list(range(len(LABELS))),
        zero_division=0,
        output_dict=True
    )
    return pd.DataFrame(report).transpose()


def save_confusion_matrix_csv(cm, out_path):
    df_cm = pd.DataFrame(cm, index=LABELS, columns=LABELS)
    df_cm.to_csv(out_path, index=True)


def make_loader(X, y, batch_size=32, shuffle=False):
    X_tensor = torch.tensor(X, dtype=torch.float32)
    y_tensor = torch.tensor(y, dtype=torch.long)

    dataset = TensorDataset(X_tensor, y_tensor)

    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=False
    )

In [4]:
def compute_class_weights(y_train, n_classes=6):
    counts = np.bincount(y_train, minlength=n_classes).astype(np.float32)

    # inverse frequency
    weights = counts.sum() / (n_classes * counts)

    # normalisasi agar rata-rata sekitar 1
    weights = weights / weights.mean()

    return torch.tensor(weights, dtype=torch.float32)

In [5]:
class HandcraftedMLP(nn.Module):
    def __init__(
        self,
        input_dim=548,
        hidden_dim=256,
        num_classes=6,
        dropout=0.30
    ):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.BatchNorm1d(hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(hidden_dim // 2, num_classes)
        )

    def forward(self, x):
        return self.net(x)

In [6]:
def run_one_epoch(model, loader, criterion, optimizer=None):
    is_train = optimizer is not None

    if is_train:
        model.train()
    else:
        model.eval()

    total_loss = 0.0
    all_preds = []
    all_targets = []

    for X_batch, y_batch in loader:
        X_batch = X_batch.to(DEVICE)
        y_batch = y_batch.to(DEVICE)

        if is_train:
            optimizer.zero_grad()

        with torch.set_grad_enabled(is_train):
            logits = model(X_batch)
            loss = criterion(logits, y_batch)

            if is_train:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
                optimizer.step()

        total_loss += loss.item() * X_batch.size(0)

        preds = torch.argmax(logits, dim=1)

        all_preds.extend(preds.detach().cpu().numpy().tolist())
        all_targets.extend(y_batch.detach().cpu().numpy().tolist())

    avg_loss = total_loss / len(loader.dataset)
    metrics = compute_metrics(np.array(all_targets), np.array(all_preds))

    return avg_loss, metrics, np.array(all_targets), np.array(all_preds)


@torch.no_grad()
def predict_model(model, loader):
    model.eval()

    all_preds = []
    all_targets = []
    all_probs = []

    for X_batch, y_batch in loader:
        X_batch = X_batch.to(DEVICE)

        logits = model(X_batch)
        probs = torch.softmax(logits, dim=1)
        preds = torch.argmax(probs, dim=1)

        all_preds.extend(preds.cpu().numpy().tolist())
        all_targets.extend(y_batch.numpy().tolist())
        all_probs.extend(probs.cpu().numpy().tolist())

    return (
        np.array(all_targets),
        np.array(all_preds),
        np.array(all_probs)
    )

In [7]:
def train_eval_mlp(
    dataset_name,
    seed,
    batch_size=32,
    lr=1e-3,
    weight_decay=1e-4,
    max_epochs=150,
    patience=20
):
    set_seed(seed)

    data = load_dataset_features(dataset_name)

    X_train = data["X_train"]
    y_train = data["y_train"]

    X_val = data["X_val"]
    y_val = data["y_val"]

    X_test = data["X_test"]
    y_test = data["y_test"]

    input_dim = X_train.shape[1]
    num_classes = len(LABELS)

    train_loader = make_loader(X_train, y_train, batch_size=batch_size, shuffle=True)
    val_loader = make_loader(X_val, y_val, batch_size=batch_size, shuffle=False)
    test_loader = make_loader(X_test, y_test, batch_size=batch_size, shuffle=False)

    model = HandcraftedMLP(
        input_dim=input_dim,
        hidden_dim=256,
        num_classes=num_classes,
        dropout=0.30
    ).to(DEVICE)

    class_weights = compute_class_weights(y_train, n_classes=num_classes).to(DEVICE)

    criterion = nn.CrossEntropyLoss(weight=class_weights)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay
    )

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="max",
        factor=0.5,
        patience=5
    )

    best_val_macro_f1 = -1.0
    best_epoch = -1
    best_state = None
    no_improve = 0

    history = []

    for epoch in range(1, max_epochs + 1):
        train_loss, train_metrics, _, _ = run_one_epoch(
            model,
            train_loader,
            criterion,
            optimizer=optimizer
        )

        val_loss, val_metrics, _, _ = run_one_epoch(
            model,
            val_loader,
            criterion,
            optimizer=None
        )

        scheduler.step(val_metrics["macro_f1"])

        row = {
            "epoch": epoch,
            "train_loss": train_loss,
            "val_loss": val_loss,
            **{f"train_{k}": v for k, v in train_metrics.items()},
            **{f"val_{k}": v for k, v in val_metrics.items()},
            "lr": optimizer.param_groups[0]["lr"]
        }
        history.append(row)

        current = val_metrics["macro_f1"]

        if current > best_val_macro_f1:
            best_val_macro_f1 = current
            best_epoch = epoch
            best_state = {
                k: v.detach().cpu().clone()
                for k, v in model.state_dict().items()
            }
            no_improve = 0
        else:
            no_improve += 1

        if epoch % 10 == 0 or epoch == 1:
            print(
                f"[{dataset_name} | seed={seed}] "
                f"Epoch {epoch:03d} | "
                f"train_loss={train_loss:.4f} | "
                f"val_loss={val_loss:.4f} | "
                f"val_macro_f1={val_metrics['macro_f1']:.4f} | "
                f"best={best_val_macro_f1:.4f}"
            )

        if no_improve >= patience:
            print(
                f"[{dataset_name} | seed={seed}] Early stopping at epoch {epoch}. "
                f"Best epoch: {best_epoch}, best val macro-F1: {best_val_macro_f1:.4f}"
            )
            break

    # Load best model
    model.load_state_dict(best_state)

    # Final predictions
    y_val_true, y_val_pred, y_val_prob = predict_model(model, val_loader)
    y_test_true, y_test_pred, y_test_prob = predict_model(model, test_loader)

    val_metrics = compute_metrics(y_val_true, y_val_pred)
    test_metrics = compute_metrics(y_test_true, y_test_pred)

    # Save outputs
    run_dir = OUT_DIR / dataset_name / f"seed_{seed}"
    run_dir.mkdir(parents=True, exist_ok=True)

    torch.save(model.state_dict(), run_dir / "mlp_model.pt")

    with open(run_dir / "training_config.json", "w") as f:
        json.dump({
            "dataset": dataset_name,
            "seed": seed,
            "batch_size": batch_size,
            "lr": lr,
            "weight_decay": weight_decay,
            "max_epochs": max_epochs,
            "patience": patience,
            "best_epoch": best_epoch,
            "best_val_macro_f1": best_val_macro_f1,
            "input_dim": input_dim,
            "num_classes": num_classes,
            "model": "HandcraftedMLP",
            "class_weighted_loss": True
        }, f, indent=2)

    pd.DataFrame(history).to_csv(run_dir / "training_history.csv", index=False)

    pd.DataFrame([{
        "dataset": dataset_name,
        "seed": seed,
        "split": "val",
        "best_epoch": best_epoch,
        **val_metrics
    }]).to_csv(run_dir / "val_metrics.csv", index=False)

    pd.DataFrame([{
        "dataset": dataset_name,
        "seed": seed,
        "split": "test",
        "best_epoch": best_epoch,
        **test_metrics
    }]).to_csv(run_dir / "test_metrics.csv", index=False)

    # Reports
    make_report_df(y_val_true, y_val_pred).to_csv(run_dir / "val_classification_report.csv")
    make_report_df(y_test_true, y_test_pred).to_csv(run_dir / "test_classification_report.csv")

    # Confusion matrices
    cm_val = confusion_matrix(y_val_true, y_val_pred, labels=list(range(len(LABELS))))
    cm_test = confusion_matrix(y_test_true, y_test_pred, labels=list(range(len(LABELS))))

    save_confusion_matrix_csv(cm_val, run_dir / "val_confusion_matrix.csv")
    save_confusion_matrix_csv(cm_test, run_dir / "test_confusion_matrix.csv")

    # Predictions
    pred_val_df = data["meta_val"].copy()
    pred_val_df["y_true"] = y_val_true
    pred_val_df["y_pred"] = y_val_pred
    pred_val_df["true_label"] = [ID_TO_LABEL[i] for i in y_val_true]
    pred_val_df["pred_label"] = [ID_TO_LABEL[i] for i in y_val_pred]

    for i, label in enumerate(LABELS):
        pred_val_df[f"prob_{label}"] = y_val_prob[:, i]

    pred_val_df.to_csv(run_dir / "val_predictions.csv", index=False)

    pred_test_df = data["meta_test"].copy()
    pred_test_df["y_true"] = y_test_true
    pred_test_df["y_pred"] = y_test_pred
    pred_test_df["true_label"] = [ID_TO_LABEL[i] for i in y_test_true]
    pred_test_df["pred_label"] = [ID_TO_LABEL[i] for i in y_test_pred]

    for i, label in enumerate(LABELS):
        pred_test_df[f"prob_{label}"] = y_test_prob[:, i]

    pred_test_df.to_csv(run_dir / "test_predictions.csv", index=False)

    row_val = {
        "dataset": dataset_name,
        "seed": seed,
        "split": "val",
        "best_epoch": best_epoch,
        **val_metrics
    }

    row_test = {
        "dataset": dataset_name,
        "seed": seed,
        "split": "test",
        "best_epoch": best_epoch,
        **test_metrics
    }

    return row_val, row_test

In [8]:
all_rows = []

for dataset_name in DATASETS:
    print("=" * 100)
    print(f"DATASET: {dataset_name.upper()}")
    print("=" * 100)

    for seed in SEEDS:
        print(f"\nTraining MLP | dataset={dataset_name} | seed={seed}")

        row_val, row_test = train_eval_mlp(
            dataset_name=dataset_name,
            seed=seed,
            batch_size=32,
            lr=1e-3,
            weight_decay=1e-4,
            max_epochs=150,
            patience=20
        )

        all_rows.append(row_val)
        all_rows.append(row_test)

        print("VAL :", {k: round(v, 4) for k, v in row_val.items() if isinstance(v, float)})
        print("TEST:", {k: round(v, 4) for k, v in row_test.items() if isinstance(v, float)})

results = pd.DataFrame(all_rows)
results.to_csv(OUT_DIR / "all_seed_results.csv", index=False)

display(results)
print("Saved:", OUT_DIR / "all_seed_results.csv")

DATASET: EMODB

Training MLP | dataset=emodb | seed=42
[emodb | seed=42] Epoch 001 | train_loss=1.4731 | val_loss=1.3085 | val_macro_f1=0.5404 | best=0.5404
[emodb | seed=42] Epoch 010 | train_loss=0.1029 | val_loss=1.1461 | val_macro_f1=0.5530 | best=0.5730
[emodb | seed=42] Epoch 020 | train_loss=0.0391 | val_loss=1.3544 | val_macro_f1=0.4896 | best=0.5730
[emodb | seed=42] Early stopping at epoch 22. Best epoch: 2, best val macro-F1: 0.5730
VAL : {'accuracy': 0.5915, 'macro_f1': 0.573, 'weighted_f1': 0.5741, 'uar': 0.5884}
TEST: {'accuracy': 0.7114, 'macro_f1': 0.7134, 'weighted_f1': 0.7122, 'uar': 0.7113}

Training MLP | dataset=emodb | seed=123
[emodb | seed=123] Epoch 001 | train_loss=1.4409 | val_loss=1.3073 | val_macro_f1=0.5162 | best=0.5162
[emodb | seed=123] Epoch 010 | train_loss=0.0931 | val_loss=1.2832 | val_macro_f1=0.4879 | best=0.5917
[emodb | seed=123] Epoch 020 | train_loss=0.0299 | val_loss=1.3790 | val_macro_f1=0.5217 | best=0.5917
[emodb | seed=123] Early stopping

,dataset,seed,split,best_epoch,accuracy,macro_f1,weighted_f1,uar
0,emodb,42,val,2,0.591549,0.573042,0.574060,0.588403
1,emodb,42,test,2,0.711409,0.713379,0.712178,0.711253
2,emodb,123,val,3,0.605634,0.591715,0.590146,0.615287
3,emodb,123,test,3,0.778523,0.783413,0.780598,0.784413
4,emodb,2024,val,2,0.633803,0.602946,0.606879,0.634363
5,emodb,2024,test,2,0.677852,0.677197,0.677448,0.671824
6,ravdess,42,val,16,0.494318,0.492211,0.484326,0.510417
7,ravdess,42,test,16,0.562500,0.547851,0.551144,0.572917
8,ravdess,123,val,11,0.465909,0.458936,0.449917,0.489583
9,ravdess,123,test,11,0.562500,0.561807,0.559665,0.578125


Saved: /content/drive/MyDrive/New Jurnal Cross/results_intra_handcrafted_mlp/all_seed_results.csv


In [9]:
metrics = ["accuracy", "macro_f1", "weighted_f1", "uar"]

summary_rows = []

for dataset_name in DATASETS:
    for split in ["val", "test"]:
        sub = results[
            (results["dataset"] == dataset_name) &
            (results["split"] == split)
        ]

        row = {
            "dataset": dataset_name,
            "split": split,
            "n_seeds": len(sub),
            "best_epoch_mean": sub["best_epoch"].mean(),
            "best_epoch_std": sub["best_epoch"].std(ddof=1),
        }

        for metric in metrics:
            row[f"{metric}_mean"] = sub[metric].mean()
            row[f"{metric}_std"] = sub[metric].std(ddof=1)

        summary_rows.append(row)

summary = pd.DataFrame(summary_rows)
summary.to_csv(OUT_DIR / "summary_mean_std.csv", index=False)

display(summary)
print("Saved:", OUT_DIR / "summary_mean_std.csv")

,dataset,split,n_seeds,best_epoch_mean,best_epoch_std,accuracy_mean,accuracy_std,macro_f1_mean,macro_f1_std,weighted_f1_mean,weighted_f1_std,uar_mean,uar_std
0,emodb,val,3,2.333333,0.577350,0.610329,0.021514,0.589235,0.015105,0.590362,0.016411,0.612685,0.023090
1,emodb,test,3,2.333333,0.577350,0.722595,0.051259,0.724663,0.053999,0.723408,0.052484,0.722497,0.057131
2,ravdess,val,3,19.000000,9.848858,0.486742,0.018264,0.482853,0.020876,0.473926,0.020854,0.503472,0.012028
3,ravdess,test,3,19.000000,9.848858,0.564394,0.003280,0.556696,0.007691,0.556044,0.004402,0.576389,0.003007
4,resd,val,3,8.666667,12.423097,0.215054,0.005376,0.213150,0.011763,0.212535,0.008974,0.214835,0.008474
5,resd,test,3,8.666667,12.423097,0.213429,0.032441,0.208880,0.041376,0.228542,0.040862,0.205268,0.026294


Saved: /content/drive/MyDrive/New Jurnal Cross/results_intra_handcrafted_mlp/summary_mean_std.csv


In [10]:
def mean_std_str(mean, std, scale=100):
    return f"{mean * scale:.2f} ± {std * scale:.2f}"


paper_rows = []

for dataset_name in DATASETS:
    sub = summary[
        (summary["dataset"] == dataset_name) &
        (summary["split"] == "test")
    ].iloc[0]

    paper_rows.append({
        "Dataset": dataset_name.upper(),
        "Accuracy": mean_std_str(sub["accuracy_mean"], sub["accuracy_std"]),
        "UAR": mean_std_str(sub["uar_mean"], sub["uar_std"]),
        "Macro-F1": mean_std_str(sub["macro_f1_mean"], sub["macro_f1_std"]),
        "Weighted-F1": mean_std_str(sub["weighted_f1_mean"], sub["weighted_f1_std"]),
        "Best Epoch": f"{sub['best_epoch_mean']:.1f} ± {sub['best_epoch_std']:.1f}",
    })

paper_table = pd.DataFrame(paper_rows)
paper_table.to_csv(OUT_DIR / "paper_table_test_mean_std.csv", index=False)

display(paper_table)
print("Saved:", OUT_DIR / "paper_table_test_mean_std.csv")

,Dataset,Accuracy,UAR,Macro-F1,Weighted-F1,Best Epoch
0,EMODB,72.26 ± 5.13,72.25 ± 5.71,72.47 ± 5.40,72.34 ± 5.25,2.3 ± 0.6
1,RAVDESS,56.44 ± 0.33,57.64 ± 0.30,55.67 ± 0.77,55.60 ± 0.44,19.0 ± 9.8
2,RESD,21.34 ± 3.24,20.53 ± 2.63,20.89 ± 4.14,22.85 ± 4.09,8.7 ± 12.4


Saved: /content/drive/MyDrive/New Jurnal Cross/results_intra_handcrafted_mlp/paper_table_test_mean_std.csv


In [11]:
per_class_rows = []

for dataset_name in DATASETS:
    for seed in SEEDS:
        report_path = OUT_DIR / dataset_name / f"seed_{seed}" / "test_classification_report.csv"
        report = pd.read_csv(report_path, index_col=0)

        for label in LABELS:
            per_class_rows.append({
                "dataset": dataset_name,
                "seed": seed,
                "class": label,
                "precision": report.loc[label, "precision"],
                "recall": report.loc[label, "recall"],
                "f1": report.loc[label, "f1-score"],
                "support": report.loc[label, "support"],
            })

per_class_df = pd.DataFrame(per_class_rows)
per_class_df.to_csv(OUT_DIR / "per_class_test_all_seeds.csv", index=False)

per_class_summary = (
    per_class_df
    .groupby(["dataset", "class"])
    .agg(
        precision_mean=("precision", "mean"),
        precision_std=("precision", "std"),
        recall_mean=("recall", "mean"),
        recall_std=("recall", "std"),
        f1_mean=("f1", "mean"),
        f1_std=("f1", "std"),
        support_mean=("support", "mean"),
    )
    .reset_index()
)

per_class_summary.to_csv(OUT_DIR / "per_class_test_summary_mean_std.csv", index=False)

display(per_class_summary)
print("Saved:", OUT_DIR / "per_class_test_summary_mean_std.csv")

,dataset,class,precision_mean,precision_std,recall_mean,recall_std,f1_mean,f1_std,support_mean
0,emodb,angry,0.701041,0.070930,0.609195,0.019909,0.651116,0.037747,29.0
1,emodb,disgust,0.781153,0.024673,0.606061,0.159631,0.676264,0.109949,22.0
2,emodb,fear,0.579371,0.061903,0.800000,0.040000,0.671669,0.055884,25.0
3,emodb,happy,0.609186,0.046399,0.625000,0.041667,0.616575,0.039108,24.0
4,emodb,neutral,0.937681,0.054362,0.818182,0.136364,0.867096,0.061865,22.0
5,emodb,sad,0.864924,0.148516,0.876543,0.021383,0.865258,0.069633,27.0
6,ravdess,angry,0.720098,0.039742,0.822917,0.036084,0.767231,0.023590,32.0
7,ravdess,disgust,0.599782,0.011439,0.687500,0.062500,0.639929,0.030926,32.0
8,ravdess,fear,0.595370,0.039901,0.385417,0.072169,0.466264,0.060738,32.0
9,ravdess,happy,0.626102,0.046018,0.541667,0.047735,0.580791,0.047096,32.0


Saved: /content/drive/MyDrive/New Jurnal Cross/results_intra_handcrafted_mlp/per_class_test_summary_mean_std.csv


In [12]:
for dataset_name in DATASETS:
    cms = []

    for seed in SEEDS:
        cm_path = OUT_DIR / dataset_name / f"seed_{seed}" / "test_confusion_matrix.csv"
        cm = pd.read_csv(cm_path, index_col=0).values
        cms.append(cm)

    cm_mean = np.mean(cms, axis=0)
    cm_std = np.std(cms, axis=0, ddof=1)

    cm_mean_df = pd.DataFrame(cm_mean, index=LABELS, columns=LABELS)
    cm_std_df = pd.DataFrame(cm_std, index=LABELS, columns=LABELS)

    ds_out = OUT_DIR / dataset_name
    cm_mean_df.to_csv(ds_out / "test_confusion_matrix_mean.csv")
    cm_std_df.to_csv(ds_out / "test_confusion_matrix_std.csv")

    print("=" * 80)
    print(dataset_name.upper())
    print("Mean confusion matrix:")
    display(cm_mean_df.round(2))

    print("Std confusion matrix:")
    display(cm_std_df.round(2))

EMODB
Mean confusion matrix:


,angry,disgust,fear,happy,neutral,sad
angry,17.67,0.00,3.00,8.33,0.00,0.00
disgust,0.00,13.33,7.00,0.00,0.67,1.00
fear,0.00,1.33,20.00,1.33,0.67,1.67
happy,7.67,0.67,0.67,15.00,0.00,0.00
neutral,0.00,1.00,1.33,0.00,18.00,1.67
sad,0.00,0.67,2.67,0.00,0.00,23.67


Std confusion matrix:


,angry,disgust,fear,happy,neutral,sad
angry,0.58,0.00,1.00,1.53,0.00,0.00
disgust,0.00,3.51,3.00,0.00,0.58,1.00
fear,0.00,0.58,1.00,0.58,0.58,1.53
happy,2.31,0.58,1.15,1.00,0.00,0.00
neutral,0.00,1.00,0.58,0.00,3.00,2.89
sad,0.00,1.15,0.58,0.00,0.00,0.58


RAVDESS
Mean confusion matrix:


...

,angry,disgust,fear,happy,neutral,sad
angry,1.15,1.15,0.58,0.58,0.00,0.58
disgust,0.58,2.00,0.58,1.15,0.00,1.15
fear,0.58,0.58,2.31,1.00,1.00,1.53
happy,0.00,1.15,1.00,1.53,1.53,1.53
neutral,1.15,0.58,0.00,0.58,0.58,1.00
sad,0.00,1.00,0.58,0.00,0.58,1.00


RESD
Mean confusion matrix:


,angry,disgust,fear,happy,neutral,sad
angry,12.00,11.67,7.33,1.33,0.33,1.33
disgust,0.67,4.67,3.00,1.00,3.33,2.33
fear,1.33,6.00,2.67,6.67,12.00,0.33
happy,2.33,4.00,6.33,8.00,1.67,0.67
neutral,1.33,3.67,11.67,1.00,0.33,2.00
sad,2.67,6.00,2.67,1.00,3.67,2.00


Std confusion matrix:


,angry,disgust,fear,happy,neutral,sad
angry,5.00,6.51,3.21,1.15,0.58,0.58
disgust,1.15,2.31,1.00,0.00,1.53,1.15
fear,0.58,3.46,2.52,5.69,4.36,0.58
happy,0.58,2.65,2.08,3.00,0.58,0.58
neutral,2.31,2.08,3.79,1.00,0.58,1.00
sad,3.79,4.36,1.53,1.00,1.53,0.00


In [13]:
SVM_DIR = BASE_PROJECT / "results_intra_handcrafted_svm"
MLP_DIR = BASE_PROJECT / "results_intra_handcrafted_mlp"

svm_summary = pd.read_csv(SVM_DIR / "summary_mean_std.csv")
mlp_summary = pd.read_csv(MLP_DIR / "summary_mean_std.csv")

svm_test = svm_summary[svm_summary["split"] == "test"].copy()
mlp_test = mlp_summary[mlp_summary["split"] == "test"].copy()

svm_test["model"] = "SVM-RBF"
mlp_test["model"] = "MLP"

compare = pd.concat([svm_test, mlp_test], ignore_index=True)

cols = [
    "model",
    "dataset",
    "accuracy_mean",
    "accuracy_std",
    "uar_mean",
    "uar_std",
    "macro_f1_mean",
    "macro_f1_std",
    "weighted_f1_mean",
    "weighted_f1_std",
]

compare = compare[cols]
compare.to_csv(MLP_DIR / "compare_svm_vs_mlp_test.csv", index=False)

display(compare)
print("Saved:", MLP_DIR / "compare_svm_vs_mlp_test.csv")

,model,dataset,accuracy_mean,accuracy_std,uar_mean,uar_std,macro_f1_mean,macro_f1_std,weighted_f1_mean,weighted_f1_std
0,SVM-RBF,emodb,0.765101,0.000000,0.771862,0.000000,0.770845,0.000000,0.766550,0.000000
1,SVM-RBF,ravdess,0.585227,0.000000,0.572917,0.000000,0.562632,0.000000,0.575213,0.000000
2,SVM-RBF,resd,0.230216,0.000000,0.216041,0.000000,0.177384,0.000000,0.201610,0.000000
3,MLP,emodb,0.722595,0.051259,0.722497,0.057131,0.724663,0.053999,0.723408,0.052484
4,MLP,ravdess,0.564394,0.003280,0.576389,0.003007,0.556696,0.007691,0.556044,0.004402
5,MLP,resd,0.213429,0.032441,0.205268,0.026294,0.208880,0.041376,0.228542,0.040862


Saved: /content/drive/MyDrive/New Jurnal Cross/results_intra_handcrafted_mlp/compare_svm_vs_mlp_test.csv
